# 🧠 Module 3: Building Autonomous Agents — Setup

This notebook prepares the knowledge base used by all subsequent notebooks in this module. You only need to run it once.

## What we're setting up

We'll create a **ChromaDB vector store** populated with OpenSearch's public documentation. This gives our agents a knowledge base to query via retrieval tools (notebook 4).

**Steps:**
1. Clone OpenSearch documentation (Apache 2.0 licensed)
2. Chunk the markdown files using LlamaIndex's sentence splitter
3. Embed chunks with Amazon Titan Embed V2
4. Store in a persistent ChromaDB collection

**Skip this notebook** if you already completed the same setup in Module 2.

## Module 3 Roadmap

| Notebook | Topic | What you'll build |
|----------|-------|-------------------|
| **1 (this)** | Setup | ChromaDB vector store |
| **2** | Memory | Chatbot with sliding-window conversation memory |
| **3** | Tools | Agent that routes to calculator + weather tools |
| **4** | Retrieval | RAG tool — agent queries the knowledge base |
| **5** | Frameworks | Same agent in LangChain, PydanticAI — interoperability |

Let's begin! 🚀

## Step 1: Download Source Data

We'll clone the OpenSearch documentation website repo (~2,000 markdown files). The `%%bash` magic runs the entire cell as a shell script.

> If you see `fatal: destination path '.' already exists` — that's fine, it means you already cloned it in a previous module. Skip to Step 2.

In [ ]:
import os
import subprocess

docs_dir = os.path.join("data", "opensearch-docs")
os.makedirs(docs_dir, exist_ok=True)

if os.path.exists(os.path.join(docs_dir, ".git")):
    print("✅ OpenSearch docs already cloned — skipping.")
else:
    print("Cloning OpenSearch documentation (this may take a minute)...")
    subprocess.run(
        ["git", "clone", "https://github.com/opensearch-project/documentation-website.git", "."],
        cwd=docs_dir, check=True
    )
    print("✅ Clone complete.")

## Step 2: Create RAG Pipeline

Now we'll chunk the markdown docs into smaller pieces suitable for embedding. We use LlamaIndex's `SentenceSplitter` for intelligent chunking that respects sentence boundaries.

**Why chunk?** Embedding models have token limits (~512 tokens), and smaller, focused chunks produce better retrieval results than entire documents.

In [ ]:
import chromadb
import boto3
from chromadb.config import Settings

# Initialize Chroma client from our persisted store
chroma_client = chromadb.PersistentClient(path="data/chroma")

# Initialize the Bedrock client
session = boto3.Session()
bedrock = session.client(service_name='bedrock-runtime')

print("✅ Client setup complete!")

The `SentenceSplitterChunkingStrategy` class below does three things:
1. **Loads** all `.md` files recursively from the docs directory
2. **Splits** them into overlapping chunks (2048 tokens, 128 overlap)
3. **Returns** a list of `RAGChunk` objects ready for ChromaDB

In [ ]:
from typing import List, Dict, Any
from pydantic import BaseModel
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Node
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline

import re

# Create a class to use instead of LlamaIndex Nodes. This way we decouple our chroma collections from LlamaIndexes
class RAGChunk(BaseModel):
    id_: str
    text: str
    metadata: Dict[str, Any] = {}


class SentenceSplitterChunkingStrategy:
    def __init__(self, input_dir: str, chunk_size: int = 256, chunk_overlap: int = 128):
        self.input_dir = input_dir
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.pipeline = self._create_pipeline()

        # Helper to get regex pattern for normalizing relative file paths.
        self.relative_path_pattern = rf"{re.escape(input_dir)}(/.*)"

    def _extract_relative_path(self, full_path):
        # Get Regex pattern
        pattern = self.relative_path_pattern
        match = re.search(pattern, full_path)
        if match:
            return match.group(1).lstrip('/')
        return None

    def _create_pipeline(self) -> IngestionPipeline:
        transformations = [
            SentenceSplitter(chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap),
        ]
        return IngestionPipeline(transformations=transformations)

    def load_documents(self) -> List[Document]:
        # If you're using a different type of file besides md, you'll want to change this. 
        return SimpleDirectoryReader(
            input_dir=self.input_dir, 
            recursive=True,
            required_exts=['.md']
        ).load_data()

    def to_ragchunks(self, nodes: List[Node]) -> List[RAGChunk]:
        chunks = []
        for node in nodes:
            metadata = {
                **node.metadata,
                'relative_path': self._extract_relative_path(node.metadata['file_path'])
            }
            # Chroma's validate_metadata rejects None values; drop any keys whose
            # value came back None (LlamaIndex often leaves creation_date /
            # last_modified_date unset, and _extract_relative_path returns None
            # when the regex doesn't match).
            metadata = {k: v for k, v in metadata.items() if v is not None}
            chunks.append(RAGChunk(id_=node.node_id, text=node.text, metadata=metadata))
        return chunks

    def process(self) -> List[RAGChunk]:
        documents = self.load_documents()
        nodes = self.pipeline.run(documents=documents)
        rag_chunks = self.to_ragchunks(nodes)
        
        print(f"Processing complete. Created {len(rag_chunks)} chunks.")
        return rag_chunks

In [ ]:
# These values were evaluated using the outputs of this notebook. https://github.com/aws-samples/genai-system-evaluation/blob/main/example-notebooks/1_Embedding_And_Chunking_Validation.ipynb
chunking_strategy = SentenceSplitterChunkingStrategy(
    input_dir="data/opensearch-docs",
    chunk_size=2048,
    chunk_overlap=128
)

# Get the nodes from the chunker.
chunks: RAGChunk = chunking_strategy.process()

## Step 3: Store in ChromaDB

We'll bulk-upload chunks into a persistent ChromaDB collection. The `ChromaDBWrapperClient` parallelizes the embedding + insertion using a thread pool (10 workers, batch size 20) to speed things up — there are thousands of chunks.

The collection persists to `data/chroma/` so subsequent notebooks can load it instantly without re-embedding.

In [ ]:
from pydantic import BaseModel
from typing import List, Dict
from abc import ABC, abstractmethod
import chromadb
from chromadb.api.types import EmbeddingFunction
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from chromadb.utils.embedding_functions import AmazonBedrockEmbeddingFunction


class RetrievalResult(BaseModel):
    id: str
    document: str
    embedding: List[float]
    distance: float
    metadata: Dict = {}


# Example of a concrete implementation
class ChromaDBWrapperClient:

    def __init__(self, chroma_client, collection_name: str, embedding_function: AmazonBedrockEmbeddingFunction):
        self.client = chroma_client
        self.collection_name = collection_name
        self.embedding_function = embedding_function

        # Create the collection
        self.collection = self._create_collection()

    def _create_collection(self):
        return self.client.get_or_create_collection(
            name=self.collection_name,
            embedding_function=self.embedding_function
        )

    def add_chunks_to_collection(self, chunks: List[RAGChunk], batch_size: int = 20, num_workers: int = 10):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(self._add_batch, batch) for batch in batches]
            for future in as_completed(futures):
                future.result()  # This will raise an exception if one occurred during the execution
        print('Finished Ingesting Chunks Into Collection')

    def _add_batch(self, batch: List[RAGChunk]):
        self.collection.add(
            ids=[chunk.id_ for chunk in batch],
            documents=[chunk.text for chunk in batch],
            metadatas=[chunk.metadata for chunk in batch]
        )

    def retrieve(self, query_text: str, n_results: int = 5) -> List[RetrievalResult]:
        # Query the collection
        results = self.collection.query(
            query_texts=[query_text],
            n_results=n_results,
            include=['embeddings', 'documents', 'metadatas', 'distances']
        )

        # Transform the results into RetrievalResult objects
        retrieval_results = []
        for i in range(len(results['ids'][0])):
            retrieval_results.append(RetrievalResult(
                id=results['ids'][0][i],
                document=results['documents'][0][i],
                embedding=results['embeddings'][0][i],
                distance=results['distances'][0][i],
                metadata=results['metadatas'][0][i] if results['metadatas'][0] else {}
            ))

        return retrieval_results

Run the cell below to embed and store all chunks. This takes **2–5 minutes** depending on your network speed (each batch calls Bedrock's Titan Embed V2 API).

In [ ]:
from chromadb.utils.embedding_functions import AmazonBedrockEmbeddingFunction

# Define some experiment variables
EMBEDDING_MODEL_ID: str = 'amazon.titan-embed-text-v2:0'
COLLECTION_NAME: str = 'opensearch-docs-rag'

# This is a handy function Chroma implemented for calling bedrock. Lets use it!
embedding_function = AmazonBedrockEmbeddingFunction(
    session=session,
    model_name=EMBEDDING_MODEL_ID
)

# Create our retrieval task. All retrieval tasks in this tutorial implement BaseRetrievalTask which has the method retrieve()
# If you'd like to extend this to a different retrieval configuration, all you have to do is create a class that that implements
# this abstract class and the rest is the same!
chroma_os_docs_collection: ChromaDBWrapperClient = ChromaDBWrapperClient(
    chroma_client = chroma_client, 
    collection_name = COLLECTION_NAME,
    embedding_function = embedding_function
)

chroma_os_docs_collection.add_chunks_to_collection(chunks)

## Step 4: Verify

Quick sanity check — query the collection and confirm we get relevant results back. If this works, you're ready for notebooks 2–5.

In [ ]:
chroma_os_docs_collection.retrieve("What is OpenSearch?")
